In [38]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 192.9 kB/s eta 0:00:0000:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 298.6 kB/s eta 0:00:0000:0100:02


In [16]:
!pip install numpy

In [17]:
# Q1 Load and inspect appointment records.
import pandas as pd
import numpy as np

df = pd.read_csv('09_hospital_appointments.csv')

print(df.head())


  Appointment_ID  Age  Department Appointment_Date  Wait_Time_Min  \
0        AP00000   18     General       2025-05-04           37.0   
1        AP00001   32  Pediatrics       2025-01-22           31.0   
2        AP00002   46     General       2025-02-26           33.0   
3        AP00003   49  Cardiology       2025-05-03           11.0   
4        AP00004   47  Cardiology       2025-05-05           26.0   

  Appointment_Type SMS_Reminder Attendance_Status  Consultation_Fee  
0        Follow-up          Yes          Attended              1200  
1        Follow-up          Yes          Attended              1200  
2              New          Yes          Attended               700  
3              New           No          Attended              1200  
4        Follow-up           No          Attended              1200  


In [8]:
# Q2 Identify and handle missing waiting times.

empty_wt = df['Wait_Time_Min'].isnull().sum()
print("Wait_Time_Min which is null or none is: ", empty_wt)

Wait_Time_Min which is null or none is:  10


In [9]:
# Q3 finding if any appointment_id repeated 
repeates = df['Appointment_ID'].duplicated().sum()
print("Appointment_ID which is repeating is: ", repeates)

Appointment_ID which is repeating is:  0


In [10]:
# Q4 Calculate total appointments and attendance-status percentages.

attendance_summary = pd.DataFrame({
    'Count': df['Attendance_Status'].value_counts(),
    'Percentage': df['Attendance_Status'].value_counts(normalize=True) * 100
})

print(attendance_summary.round(2))

                   Count  Percentage
Attendance_Status                   
Attended             397        79.4
No-show               60        12.0
Cancelled             43         8.6


In [ ]:
# Q5 Calculate average waiting time by department.

# Used groupby to sort department and mean function to compute the mean

avg_waiting_time = df.groupby('Department')['Wait_Time_Min'].mean()

print(avg_waiting_time)

Department
Cardiology     25.814433
Dermatology    29.902913
General        27.115789
Orthopedics    27.133333
Pediatrics     30.428571
Name: Wait_Time_Min, dtype: float64


In [ ]:
# Q6 Calculate no-show percentage by department
no_show_percentage = (
    df.assign(No_Show=df['Attendance_Status'].eq('No-show'))
      .groupby('Department')['No_Show']
      .mean() * 100
)

print(no_show_percentage.sort_values(ascending=False))

# Department with highest no-show percentage
print("\nHighest No-Show Department:")
print(no_show_percentage.idxmax())

Department
Pediatrics     14.953271
Dermatology    14.285714
Orthopedics    13.043478
General         9.183673
Cardiology      8.163265
Name: No_Show, dtype: float64

Highest No-Show Department:
Pediatrics


In [11]:
# Q7 Compare no-show rates for SMS reminder Yes and No.

# Calculate no-show percentage for SMS reminders
sms_no_show = (
    df.assign(No_Show=df['Attendance_Status'].eq('No-show'))
      .groupby('SMS_Reminder')['No_Show']
      .mean() * 100
)

print("No-show Percentage by SMS Reminder:")
print(sms_no_show.round(2))

No-show Percentage by SMS Reminder:
SMS_Reminder
No     13.44
Yes    10.53
Name: No_Show, dtype: float64


In [12]:
# Q8 Calculate average consultation fee by department.

avg_amt = df.groupby('Department')['Consultation_Fee'].mean()

print(avg_amt)

Department
Cardiology     688.775510
Dermatology    748.571429
General        777.551020
Orthopedics    689.130435
Pediatrics     748.598131
Name: Consultation_Fee, dtype: float64


In [14]:
# Q9 Analyze appointment volume by date and department.

appointment_volume = df.pivot_table(
    index='Appointment_Date',
    columns='Department',
    values='Appointment_ID',
    aggfunc='count',
    fill_value=0
)

print(appointment_volume)

Department        Cardiology  Dermatology  General  Orthopedics  Pediatrics
Appointment_Date                                                           
2025-01-01                 2            0        0            0           1
2025-01-02                 1            0        1            0           0
2025-01-03                 1            2        1            0           0
2025-01-04                 0            2        1            0           0
2025-01-05                 0            0        0            0           1
...                      ...          ...      ...          ...         ...
2025-06-23                 1            1        2            0           0
2025-06-26                 2            0        0            0           1
2025-06-27                 0            0        0            0           1
2025-06-28                 3            0        0            1           0
2025-06-29                 1            0        0            1           1

[163 rows x

In [15]:
# Q10 Compare waiting times for new and follow-up appointments.

avg_waiting = df.groupby('Appointment_Type')['Wait_Time_Min'].mean()

print("Average Waiting Time:")
print(avg_waiting)

Average Waiting Time:
Appointment_Type
Follow-up    28.254167
New          28.064000
Name: Wait_Time_Min, dtype: float64


In [18]:
# Q11  Use NumPy to calculate median, quartiles and standard deviation of waiting time.

waiting_time = df['Wait_Time_Min'].dropna()

# Calculate median
median = np.median(waiting_time)

# Calculate quartiles
Q1 = np.percentile(waiting_time, 25)
Q2 = np.percentile(waiting_time, 50)
Q3 = np.percentile(waiting_time, 75)

# Calculate standard deviation
std_dev = np.std(waiting_time)

# Display results
print("Median:", median)
print("First Quartile (Q1):", Q1)
print("Second Quartile (Q2):", Q2)
print("Third Quartile (Q3):", Q3)
print("Standard Deviation:", std_dev)

Median: 28.0
First Quartile (Q1): 18.0
Second Quartile (Q2): 28.0
Third Quartile (Q3): 38.0
Standard Deviation: 14.418589191932554


In [19]:
# Q12 Identify unusually long waiting times using IQR.


import numpy as np

# Remove missing values
waiting_time = df['Wait_Time_Min'].dropna()

# Calculate Q1 and Q3
Q1 = np.percentile(waiting_time, 25)
Q3 = np.percentile(waiting_time, 75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate upper limit
upper_limit = Q3 + (1.5 * IQR)

# Identify unusually long waiting times
long_waiting = df[df['Wait_Time_Min'] > upper_limit]

# Display results
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Upper Limit:", upper_limit)

print("\nUnusually Long Waiting Times:")
print(long_waiting)


Q1: 18.0
Q3: 38.0
IQR: 20.0
Upper Limit: 68.0

Unusually Long Waiting Times:
    Appointment_ID  Age   Department Appointment_Date  Wait_Time_Min  \
121        AP00121   78  Orthopedics       2025-05-17           69.0   
226        AP00226   21  Orthopedics       2025-03-27           71.0   

    Appointment_Type SMS_Reminder Attendance_Status  Consultation_Fee  
121              New           No           No-show              1200  
226        Follow-up           No          Attended              1200  


In [20]:
# Q13 Create waiting-time categories.


import pandas as pd

# Create waiting-time categories
bins = [0, 15, 30, 45, float('inf')]

labels = ['Short', 'Moderate', 'Long', 'Very Long']

df['Waiting_Category'] = pd.cut(
    df['Wait_Time_Min'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Display results
print(df[['Wait_Time_Min', 'Waiting_Category']])

# Count appointments in each category
print("\nWaiting Time Category Counts:")
print(df['Waiting_Category'].value_counts())


     Wait_Time_Min Waiting_Category
0             37.0             Long
1             31.0             Long
2             33.0             Long
3             11.0            Short
4             26.0         Moderate
..             ...              ...
495            NaN              NaN
496           20.0         Moderate
497           10.0            Short
498           22.0         Moderate
499           20.0         Moderate

[500 rows x 2 columns]

Waiting Time Category Counts:
Waiting_Category
Moderate     189
Long         145
Short         95
Very Long     61
Name: count, dtype: int64


In [21]:
# Q14 Calculate estimated total consultation fees for attended appointments.

print(df.loc[df['Attendance_Status'] == 'Attended', 'Consultation_Fee'].sum())

296200


Q15 
1. Reduce Waiting Time: Increase staff availability and improve appointment scheduling in departments with higher average waiting times.

2. Reduce No Shows: Send SMS reminders to patients before their appointments to improve attendance rates.

3. Improve Department Management: Allocate doctors and medical staff according to appointment volume in each department.

4. Manage Appointment Scheduling: Avoid overcrowding by distributing appointments evenly throughout the day.

5. Monitor Performance: Regularly analyze waiting times, no-show rates and consultation fees to improve hospital efficiency and patient satisfaction.